# as-strided-noncontig-source — ex9: diagonal extraction via stride manipulation

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-noncontig-source`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## strides and non-contiguity — quick refresher

**Stride** = number of *elements* (not bytes) to advance one step along an axis. A contiguous `(H, W)` float tensor has stride `(W, 1)`.

**`torch.as_strided(input, size, stride)`** builds a zero-copy view at the exact (shape, stride) you specify. It bypasses safety checks — overlapping windows, out-of-bounds offsets, the works. Powerful, dangerous, and the foundation of rolling-window tricks, im2col, and stride-based broadcasting hacks.

**`.contiguous()`** materializes a row-major copy if the current strides aren't already row-major. Required before `.view()`; optional but often a perf-vs-memory trade-off otherwise.

### Exercise 9 — diagonal extraction via stride manipulation

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Use stride arithmetic to extract the main diagonal of a square matrix as a zero-copy 1-D view, and visualize the access pattern.
> Keywords: diagonal, as_strided, stride-arithmetic, visualization
> ```

**KCs targeted:** `strides-anatomy`, `as-strided-arbitrary-view`

Implement `ex9_diagonal_via_strided(m)`. Given a square `(N, N)` tensor `m`, return a 1-D view of length `N` containing `[m[0,0], m[1,1], ..., m[N-1,N-1]]` — the main diagonal — built **using only `t.as_strided`**.

Key insight: if `m` has stride `(sR, sC)`, then advancing one step along the diagonal means moving one row *and* one column — a single offset of `sR + sC` elements in storage. So the diagonal view has shape `(N,)` and stride `(sR + sC,)`.

The test verifies values + zero-copy aliasing, then visualizes the access pattern: a heatmap of `m` with the diagonal cells highlighted, showing exactly which storage positions your strided view reads.

In [ ]:
def ex9_diagonal_via_strided(m: Tensor) -> Tensor:
    """Return the main diagonal of square m as a zero-copy 1-D view."""
    raise NotImplementedError()


def _test_ex9():
    import numpy as np

    N = 5
    m = t.arange(N * N, dtype=t.float32).reshape(N, N)
    diag = ex9_diagonal_via_strided(m)

    # Shape and value checks.
    assert diag.shape == (N,), f'expected ({N},), got {tuple(diag.shape)}'
    expected = t.tensor([m[i, i].item() for i in range(N)])
    assert t.equal(diag, expected), f'diagonal mismatch: {diag} vs {expected}'

    # Zero-copy: must share storage with m.
    assert diag.data_ptr() == m.data_ptr(), 'diagonal must be a view, not a copy'

    # Stride sanity: should be sR + sC for contiguous m, that's N + 1 = 6.
    sR, sC = m.stride()
    assert diag.stride() == (sR + sC,), f'expected stride ({sR + sC},), got {diag.stride()}'

    # Cross-check against t.diagonal as ground truth.
    assert t.equal(diag, m.diagonal()), 'must agree with t.diagonal'

    # Mutation propagates back through the view.
    diag2 = ex9_diagonal_via_strided(m)
    before = m[2, 2].item()
    diag2[2] = -7.0
    assert m[2, 2].item() == -7.0, 'mutation through diag view should hit m[i,i]'
    m[2, 2] = before  # restore for the plot

    # Visualize: heatmap of m with diagonal cells outlined.
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.imshow(m.numpy(), cmap='Blues')
    for i in range(N):
        for j in range(N):
            color = 'red' if i == j else 'black'
            weight = 'bold' if i == j else 'normal'
            ax.text(j, i, f'{int(m[i, j].item())}', ha='center', va='center',
                    color=color, fontweight=weight)
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    ax.set_title(f'diag = as_strided(m, ({N},), ({sR + sC},))  -- red cells are the view')
    fig.tight_layout()
    plt.show()

    print(f'm.stride() = {m.stride()}')
    print(f'diag.stride() = {diag.stride()}  (= sR + sC = {sR + sC})')
    print(f'diag = {diag.tolist()}')
    _dd_passed.add('ex9')
    print("ex9 ✓")

_test_ex9()

<details><summary>Solution</summary>

```python
def ex9_diagonal_via_strided(m: Tensor) -> Tensor:
    N = m.shape[0]
    sR, sC = m.stride()
    return t.as_strided(m, size=(N,), stride=(sR + sC,))
```

**Why `sR + sC` is the diagonal step.** From `m[i, i]` to `m[i+1, i+1]` you move one row down (`+sR` elements in storage) *and* one column right (`+sC` elements). The diagonal view is just the source storage sampled every `sR + sC` elements.

**Off-diagonals.** For the `k`-th diagonal above the main, start your view at offset `k * sC` (use `t.as_strided(m[..., k:], ...)` or pass an explicit `storage_offset`) with shape `(N - k,)` and stride `(sR + sC,)`. Below the main, swap roles. This is the entire trick behind banded-matrix routines.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()